In [ ]:
#import libraries
import pandas as pd
import numpy as np


In [ ]:
train_df_fe=pd.read_csv("D:\\Project\\Desktop Files\\Clickstream\\Notebooks\\feature_engineered_data.csv")

In [ ]:
train_df_fe.columns

Index(['month', 'day', 'order', 'page1_main_category', 'page2_clothing_model',
       'location', 'model_photography', 'price', 'price_2', 'page',
       'country_grouped', 'colour_grouped', 'day_grouped'],
      dtype='object')

In [ ]:
#drop unnecessary columns
train_df_fe.drop(["day"], axis=1)

In [ ]:
#Classification dataset

X_cls=train_df_fe.drop(columns=["price_2","price"])
y_cls=train_df_fe["price_2"]

In [ ]:
#

y_cls = y_cls.replace({
    1: 1,
    2: 0
})

y_cls.value_counts()

In [ ]:
#Regression dataset
X_reg=train_df_fe.drop(columns=["price", "price_2"])
y_reg=train_df_fe["price"]

In [ ]:
#Train_test_split

#Classification

from sklearn.model_selection import train_test_split

X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls,
    y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
) 

#Regression
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg,
    y_reg,
    test_size=0.2,
    random_state=42
)

In [ ]:
#Model frequency

model_freq_cls = X_train_cls["page2_clothing_model"].value_counts()

#Apply mapping

#Classification train
X_train_cls["page2_clothing_model_freq"] = (
    X_train_cls["page2_clothing_model"].map(model_freq_cls)
)

#Classification Test set

X_test_cls["page2_clothing_model_freq"] = (
    X_test_cls["page2_clothing_model"].map(model_freq_cls)
    .fillna(0)
)

#Model frequency Regression

model_freq_reg = X_train_reg["page2_clothing_model"].value_counts()

#Apply mapping 

#Regression train
X_train_reg["page2_clothing_model_freq"] = (
    X_train_reg["page2_clothing_model"].map(model_freq_reg)
)

#Regression Test set
X_test_reg["page2_clothing_model_freq"] = (
    X_test_reg["page2_clothing_model"].map(model_freq_reg)
    .fillna(0)
)



In [ ]:
#Numerical features

numerical_features = [
    "order",
    "page2_clothing_model_freq"
]

#Categorical features

categorical_features = [
    "month",
    "day_grouped",
    "country_grouped",
    "colour_grouped",
    "page1_main_category",
    "location",
    "model_photography",
    "page"
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from sklearn.preprocessing import StandardScaler,OneHotEncoder


In [ ]:
#pipelines for numerical and categorical features

num_pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler())
    ]
)

cat_pipeline = Pipeline(
    steps=[
        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ]
)

In [ ]:
#preprocessor 

preprocessor = ColumnTransformer(
    transformers=[
        ("num", num_pipeline, numerical_features),
        ("cat", cat_pipeline, categorical_features)
    ]
)

In [ ]:
#Fit_Processor for classification 

X_train_cls_preprocessed = preprocessor.fit_transform(X_train_cls)
X_test_cls_preprocessed = preprocessor.transform(X_test_cls)

#Fit_Processor for regression
X_train_reg_preprocessed = preprocessor.fit_transform(X_train_reg)
X_test_reg_preprocessed = preprocessor.transform(X_test_reg)

In [ ]:
#Checking shapes of processed data

print(X_train_cls_preprocessed.shape)
print(X_test_cls_preprocessed.shape)

print(X_train_reg_preprocessed.shape)
print(X_test_reg_preprocessed.shape)

In [ ]:
#Saving the preprocessed data for later use
import joblib
joblib.dump(X_train_cls_preprocessed, "X_train_cls_preprocessed.pkl")
joblib.dump(X_test_cls_preprocessed, "X_val_cls_preprocessed.pkl")
joblib.dump(X_train_reg_preprocessed, "X_train_reg_preprocessed.pkl")
joblib.dump(X_test_reg_preprocessed, "X_val_reg_preprocessed.pkl")

# saving preprocessor
joblib.dump(preprocessor_cls, "preprocessor.pkl")